# Final Data Science Project – End-to-End Case Study

## Problem Statement
Build a complete data-science pipeline to predict a quantitative disease-progression measure using patient-related features. The project covers data preparation, exploratory data analysis (EDA), preprocessing, baseline modeling, evaluation, and conclusions.

## Objectives
- Load and inspect a public dataset.
- Clean and validate the data.
- Explore distributions and relationships using EDA.
- Split data into training and testing sets.
- Train a baseline Linear Regression model and a Random Forest Regressor.
- Evaluate models using MAE, RMSE and R².
- Identify limitations and possible improvements.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_columns', None)


## 1. Load Dataset

In [ ]:
df = pd.read_csv('diabetes_dataset.csv')
print('Shape:', df.shape)
display(df.head())


## 2. Data Preparation

In [ ]:
print(df.info())
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())
print('\nSummary statistics:')
display(df.describe().T)

# Remove exact duplicate rows if any exist.
df = df.drop_duplicates().reset_index(drop=True)

# Ensure all columns are numeric for modeling.
df = df.apply(pd.to_numeric, errors='coerce')
df = df.dropna().reset_index(drop=True)
print('Shape after cleaning:', df.shape)


## 3. Exploratory Data Analysis (EDA)

In [ ]:
target = 'target'

# Target distribution
plt.figure(figsize=(8,5))
plt.hist(df[target], bins=25)
plt.xlabel('Disease progression target')
plt.ylabel('Frequency')
plt.title('Target Distribution')
plt.show()

# Correlation with target
corr = df.corr(numeric_only=True)[target].sort_values(ascending=False)
display(corr.to_frame('Correlation with target'))

plt.figure(figsize=(9,5))
corr.drop(target).sort_values().plot(kind='barh')
plt.xlabel('Correlation')
plt.title('Feature Correlation with Target')
plt.show()


## 4. Define Features and Target

In [ ]:
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))


## 5. Modeling

In [ ]:
models = {
    'Linear Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    )
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    results.append([name, mae, rmse, r2])

results_df = pd.DataFrame(results, columns=['Model','MAE','RMSE','R2'])
display(results_df.sort_values('R2', ascending=False))


## 6. Actual vs Predicted

In [ ]:
best_model_name = results_df.sort_values('R2', ascending=False).iloc[0]['Model']
best_pred = predictions[best_model_name]

plt.figure(figsize=(7,6))
plt.scatter(y_test, best_pred, alpha=0.7)
mn, mx = min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())
plt.plot([mn, mx], [mn, mx], linestyle='--')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.title(f'Actual vs Predicted – {best_model_name}')
plt.show()

print('Best model:', best_model_name)


## 7. Conclusion

The project demonstrates a complete data-science workflow from raw data inspection to model evaluation. Linear Regression provides a simple baseline, while Random Forest provides a nonlinear alternative. MAE and RMSE measure prediction error, while R² measures explained variance. The model with the strongest test-set R² can be selected as the baseline winner, but model selection should also consider interpretability and generalization.

### Limitations
- The dataset is relatively small, so test results can vary with the train/test split.
- A single holdout split is not enough to establish production-level performance.
- The dataset is intended for educational/research use; predictions should not be interpreted as clinical advice.
- Hyperparameter tuning and cross-validation can improve the analysis.

### Future Scope
- Apply k-fold cross-validation.
- Tune Random Forest and compare additional algorithms such as Gradient Boosting.
- Perform feature-importance and error analysis.
- Package the selected model into a simple Streamlit or Flask application.
